# 首板低开策略回测

基于聚宽文章：https://www.joinquant.com/post/44901

策略逻辑：
1. **选股范围**：排除新股(250天)、ST、科创板、北交所
2. **昨日首板**：昨日涨停且非连板
3. **低开过滤**：开盘涨幅 -4% 到 -3%
4. **相对位置**：rp = (close-low)/(high-low) ≤ 0.6
5. **买入**：9:30 开盘价买入
6. **卖出**：
   - 11:28：未涨停且盈利 → 止盈
   - 14:50：未涨停 → 止损

In [ ]:
%reload_ext autoreload
%autoreload 2
from my_utils.fun import *
import polars as pl
import pandas as pd
import datetime as dt
import time

logging = get_logger(log_file='log/首板低开策略.log', inherit=False)

start_date = dt.date(2021, 1, 1)
end_date = dt.datetime.today()

In [3]:
# 1. 读取日线数据
stock_data = read_day_data(start_date=start_date, end_date=end_date)

# 2. 标记涨跌停状态
stock_data = mark_limit_status(stock_data)
stock_data = mark_limit_desc(stock_data)

# 3. 计算相对位置 rp = (close - low) / (high - low)
stock_data = stock_data.with_columns(
    ((pl.col('close') - pl.col('low')) / (pl.col('high') - pl.col('low'))).alias('relative_position')
)

# 4. 计算开盘涨幅 open_pct
stock_data = stock_data.with_columns(
    ((pl.col('open') - pl.col('pre_close')) / pl.col('pre_close') * 100).alias('open_pct')
)

print('数据加载完成')
print(f'数据范围: {start_date} 至 {end_date}')
print(f'股票数量: {stock_data.select("code").n_unique()}')

数据加载完成
数据范围: 2021-01-01 至 2026-04-25 20:45:14.736030
股票数量: 5487


In [4]:
# 5. 计算连板股票（严格复现聚宽逻辑）
def get_continue_limit_stocks(df: pl.DataFrame, watch_days: int = 10) -> set:
    """
    获取连板股票集合
    聚宽逻辑：遍历d从2到watch_days，检查每个窗口是否每天都涨停
    如果某窗口的涨停天数等于窗口大小，说明每天都涨停，即连板
    """
    continue_limit_stocks = set()
    
    for code in df.select('code').unique().to_series():
        stock_df = df.filter(pl.col('code') == code).sort('trading_date')
        n = stock_df.height
        
        if n < watch_days:
            continue
        
        is_limit_list = stock_df.select('is_limit_up').to_series().to_list()
        
        # 遍历每个可能的窗口起点
        for start_idx in range(n - watch_days + 1):
            window = is_limit_list[start_idx:start_idx + watch_days]
            
            # 检查是否存在连续N天涨停（从2到watch_days）
            for d in range(2, watch_days + 1):
                sub_window = window[:d]
                if all(sub_window):  # 如果前d天每天都涨停
                    continue_limit_stocks.add(code)
                    break  # 找到一个连板就break，避免重复添加
    
    return continue_limit_stocks

# 获取连板股票
logging.info('开始计算连板股票...')
continue_limit_stocks = get_continue_limit_stocks(stock_data, watch_days=10)
print(f'连板股票数量: {len(continue_limit_stocks)}')
print(f'连板股票示例: {list(continue_limit_stocks)[:5]}')

开始计算连板股票...


连板股票数量: 4167
连板股票示例: ['SHSE.603000', 'SZSE.002290', 'SHSE.600540', 'SZSE.000505', 'SHSE.603021']


In [5]:
# 6. 生成买入信号
# 确保按股票和日期排序
stock_data = stock_data.sort(['code', 'trading_date'])

# 计算前一日的关键字段
stock_data = stock_data.with_columns([
    pl.col('is_limit_up').shift(1).over('code').alias('prev_is_limit_up'),
    pl.col('limit_desc').shift(1).over('code').alias('prev_limit_desc'),
    pl.col('relative_position').shift(1).over('code').alias('prev_relative_position'),
])

# 判断昨日是否是连板（检查是否在连板集合中）
stock_data = stock_data.with_columns(
    pl.col('code').is_in(pl.lit(list(continue_limit_stocks))).alias('is_continue_limit')
)

# 信号参数（严格按聚宽）
params_dict = {
    'open_pct_low': -4.0,    # 开盘涨幅下限 -4%
    'open_pct_high': -3.0,   # 开盘涨幅上限 -3%（实际0.96-0.97对应-4%到-3%）
    'rp_threshold': 0.6,     # 相对位置阈值
}

# 生成信号
# 买入条件：
# 1. 昨日涨停
# 2. 昨日不是连板
# 3. 今日低开在-4%到-3%之间
# 4. 昨日相对位置<=0.6

stock_data = stock_data.with_columns(
    signal=pl.when(
        # 过滤条件：非ST
        ~(pl.col('is_st')) &
        # 过滤条件：非科创板(688)、非北交所(430/8)、非创业板(30)
        ~(pl.col('code').str.split('.').list[1].str.starts_with('688') | 
          pl.col('code').str.split('.').list[1].str.starts_with('430') |
          pl.col('code').str.split('.').list[1].str.starts_with('30') |
          pl.col('code').str.split('.').list[1].str.starts_with('8')) &
        # 1. 昨日涨停
        (pl.col('prev_is_limit_up') == True) &
        # 2. 昨日不是连板
        (pl.col('is_continue_limit') == False) &
        # 3. 今日低开在-4%到-3%之间
        (pl.col('open_pct') >= params_dict['open_pct_low']) &
        (pl.col('open_pct') <= params_dict['open_pct_high']) &
        # 4. 昨日相对位置<=0.6
        (pl.col('prev_relative_position') <= params_dict['rp_threshold'])
    ).then(1).otherwise(0)
)

# 筛选信号股票
信号文件 = stock_data.filter(pl.col('signal') == 1)
print(f'买入信号参数: {params_dict}')
print(f'买入信号数量: {信号文件.height}')
if 信号文件.height > 0:
    print(f'买入信号日期范围: {信号文件.select("trading_date").min().item()} 至 {信号文件.select("trading_date").max().item()}')

买入信号参数: {'open_pct_low': -4.0, 'open_pct_high': -3.0, 'rp_threshold': 0.6}
买入信号数量: 0


In [6]:
# 7. 查看信号样例
if 信号文件.height > 0:
    sample = 信号文件.select(['trading_date', 'code', 'name', 'open_pct', 'prev_limit_desc', 'prev_relative_position']).head(10)
    print('信号样例:')
    print(sample)

In [7]:
# 8. 定义严格匹配聚宽的交易函数
def first_board_low_open_trade(code_list, trade_date, fee_rate=0.0005, need_adj=True):
    """
    严格复现聚宽首板低开策略的交易逻辑
    买入: 9:30 开盘价
    卖出: 
        - 11:28: 未涨停且盈利 -> 止盈
        - 14:50: 未涨停 -> 止损
    """
    from datetime import datetime, timedelta, time
    
    start_process_time = time.time()
    start_date = trade_date
    end_date = start_date + timedelta(days=15)
    
    # 1. 获取数据
    try:
        stock_data = read_day_data(start_date, end_date, code_list)
        mins_data = read_min_data(start_date, end_date, code_list)
        
        if need_adj:
            adj_data = read_day_data(start_date, end_date, code_list, file_path='ts_adj')
            stock_data = stock_data.join(
                adj_data[['trading_date', 'code', 'adj_factor']],
                on=['trading_date', 'code'],
                how='left',
            )
            stock_data = stock_data.rename({'adj_factor': 'adj'})
        
        fields_to_merge = ['pre_close', 'limit_up', 'limit_down']
        if need_adj:
            fields_to_merge += ['adj']
        mins_data = mins_data.join(
            stock_data[['trading_date', 'code'] + fields_to_merge],
            on=['trading_date', 'code'],
            how='left',
        )
        mins_data = mins_data.drop_nulls(subset=['open', 'close', 'pre_close', 'limit_up', 'limit_down'])
    except Exception as e:
        logging.info(f'获取{code_list}数据失败: {str(e)}')
        return None
    
    result = []
    
    for code in code_list:
        code_mins_data = mins_data.filter(pl.col('code') == code)
        
        trade_info = {
            'code': code,
            'buy_time': None,
            'buy_price': None,
            'sell_time': None,
            'sell_price': None,
            'profit': None,
            'holding_days': None,
            'sell_reason': None
        }
        
        if code_mins_data.height == 0:
            continue
        
        trading_date_list = sorted(code_mins_data['trading_date'].unique().to_list())
        if len(trading_date_list) < 2:
            continue
        
        buy_date = trading_date_list[0]
        
        # 2.1 买入: 9:30 开盘价
        buy_data = code_mins_data.filter(
            (pl.col('trading_date') == buy_date) &
            (pl.col('datetime').dt.hour() == 9) &
            (pl.col('datetime').dt.minute() == 30)
        )
        
        if buy_data.height == 0:
            continue
        
        buy_price = buy_data['open'].to_list()[0]
        buy_adj = buy_data['adj'].to_list()[0] if 'adj' in buy_data.columns else 1.0
        trade_info['buy_time'] = datetime.combine(buy_date, time(9, 30))
        trade_info['buy_price'] = buy_price
        buy_date_index = trading_date_list.index(buy_date)
        
        # 2.2 卖出逻辑
        target_times = [time(11, 28), time(14, 50)]  # 严格按聚宽的时间点
        end_date = trading_date_list[-1]
        sell_triggered = False
        
        # 遍历T+1及之后的日子
        for i, single_date in enumerate(trading_date_list[1:]):
            full_days = i + 1
            
            # 获取当天在目标时间点的数据
            code_daily_data = code_mins_data.filter(
                (pl.col('trading_date') == single_date) &
                (pl.col('datetime').dt.time().is_in(target_times))
            )
            
            if code_daily_data.height == 0:
                continue
            
            # 获取当日涨停价和前收盘价
            limit_up = code_daily_data['limit_up'].to_list()[0]
            pre_close = code_daily_data['pre_close'].to_list()[0]
            
            # 检查每个时间点
            for row in code_daily_data.iter_rows(named=True):
                current_price = row['open']
                current_time = row['datetime'].time()
                adj = row['adj'] if 'adj' in row.keys() else 1.0
                
                # 计算持有天数
                same_day_ratio = calculate_time_ratio(current_time)
                total_holding_days = round(full_days + same_day_ratio, 2)
                
                # ========== 卖出条件（严格按聚宽）==========
                # 
                # 11:28 止盈: 未涨停(current_price < limit_up) 且 盈利(current_price > buy_price)
                # 14:50 止损: 未涨停(current_price < limit_up)
                
                is_not_limit_up = current_price < limit_up
                is_profitable = current_price > trade_info['buy_price']
                
                # 11:28 止盈条件
                if current_time == time(11, 28) and is_not_limit_up and is_profitable:
                    buy_price_fee = trade_info['buy_price'] * buy_adj * (1 + fee_rate)
                    sell_price_fee = current_price * adj * (1 - fee_rate)
                    profit = (sell_price_fee - buy_price_fee) / buy_price_fee * 100
                    trade_info.update({
                        'sell_time': datetime.combine(row['trading_date'], current_time),
                        'sell_price': current_price,
                        'profit': profit,
                        'holding_days': total_holding_days,
                        'sell_reason': '11:28止盈'
                    })
                    sell_triggered = True
                    break
                
                # 14:50 止损条件（只要未涨停就卖，不考虑是否盈利）
                if current_time == time(14, 50) and is_not_limit_up:
                    buy_price_fee = trade_info['buy_price'] * buy_adj * (1 + fee_rate)
                    sell_price_fee = current_price * adj * (1 - fee_rate)
                    profit = (sell_price_fee - buy_price_fee) / buy_price_fee * 100
                    trade_info.update({
                        'sell_time': datetime.combine(row['trading_date'], current_time),
                        'sell_price': current_price,
                        'profit': profit,
                        'holding_days': total_holding_days,
                        'sell_reason': '14:50止损'
                    })
                    sell_triggered = True
                    break
            
            if sell_triggered:
                break
        
        # 如果未触发卖出，最后一天收盘价卖出
        if not trade_info['sell_time']:
            final_day_data = code_mins_data.filter(pl.col('trading_date') == end_date)
            adj = final_day_data['adj'][0] if 'adj' in final_day_data.columns else 1.0
            
            if final_day_data.height > 0:
                sell_price = final_day_data['close'].to_list()[-1]
                final_time = final_day_data['datetime'].to_list()[-1].time()
                
                full_days = trading_date_list.index(end_date) - buy_date_index
                same_day_ratio = calculate_time_ratio(final_time)
                total_holding_days = round(full_days + same_day_ratio, 2)
                
                if trade_info['buy_price']:
                    buy_price_fee = trade_info['buy_price'] * buy_adj * (1 + fee_rate)
                    sell_price_fee = sell_price * adj * (1 - fee_rate)
                    profit = (sell_price_fee - buy_price_fee) / buy_price_fee * 100
                    trade_info['profit'] = profit
                
                trade_info.update({
                    'sell_time': datetime.combine(end_date, final_time),
                    'sell_price': sell_price,
                    'holding_days': total_holding_days,
                    'sell_reason': '最后卖出'
                })
        
        result.append(trade_info)
    
    return result

In [8]:
# 9. 运行回测
from my_utils.trade_fun import cal_trade_info, calculate_time_ratio

start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str = end_date.strftime('%Y-%m-%d')
logging.info(f'回测时间区间: {start_date_str} 至 {end_date_str}')

# 使用新的交易函数
result_df, merged_df = cal_trade_info(
    信号文件, 
    trade_fun=first_board_low_open_trade, 
    start_date=start_date_str, 
    end_date=end_date_str
)

print(f'回测完成，共 {merged_df.height} 条交易记录')

回测时间区间: 2021-01-01 至 2026-04-25
没有符合条件的回测日期


回测完成，共 0 条交易记录


In [9]:
# 10. 回测结果汇报
from my_utils.trade_fun import report_backtest_full

logging.info('=' * 50)
logging.info('首板低开策略回测结果（不控制仓位）')
logging.info('=' * 50)

back_result = report_backtest_full(
    merged_df.to_pandas(), 
    start_date=start_date_str, 
    end_date=end_date_str,
    profit_col='profit',
    plot=True
)

首板低开策略回测结果（不控制仓位）


KeyError: 'buy_time'

In [ ]:
# 11. 仓位控制回测（0.4仓位）
merged_df = merged_df.with_columns(
    (pl.col('profit') * 0.4).alias('weight_profit')
)

logging.info('=' * 50)
logging.info('仓位控制回测结果（40%仓位）')
logging.info('=' * 50)

back_result_weighted = report_backtest_full(
    merged_df.to_pandas(), 
    start_date=start_date_str, 
    end_date=end_date_str,
    profit_col='weight_profit',
    plot=True
)

In [ ]:
# 12. 真实资金回测（使用 my_backtester）
from my_backtester.my_backtester import Backtester
import pandas as pd

# 准备订单数据
交割数据 = merged_df.to_pandas()
orders_list = []

for _, row in 交割数据.iterrows():
    buy_order = {
        'datetime': pd.to_datetime(row['buy_time']),
        'code': row['code'],
        'direction': 1,
        'price': row['buy_price'],
        'cash_ratio': 0.4,
        'buy_time': row['buy_time'],
        'sell_time': row['sell_time'],
    }
    sell_order = {
        'datetime': pd.to_datetime(row['sell_time']),
        'code': row['code'],
        'direction': -1,
        'price': row['sell_price'],
        'cash_ratio': 0.4,
        'buy_time': row['buy_time'],
        'sell_time': row['sell_time'],
    }
    orders_list.append(buy_order)
    orders_list.append(sell_order)

orders_df = pd.DataFrame(orders_list)
print(f'订单总数: {len(orders_df)}')
print(f'买入订单: {len(orders_df[orders_df["direction"] == 1])}')
print(f'卖出订单: {len(orders_df[orders_df["direction"] == -1])}')

In [ ]:
# 13. 运行真实资金回测
backtester = Backtester(
    orders=orders_df, 
    initial_cash=10000000,
    commission=0.001,
    slippage=0.001
)

资金结果 = backtester.run(
    start_time=start_date_str, 
    end_time=end_date_str
)

持仓结果 = backtester.pos_log
交易记录 = backtester.trade_log

print('回测完成！')

In [ ]:
# 14. 汇报真实资金回测结果
metrics_df, fig = backtester.report(
    return_method='compound', 
    plot=True
)

print('\n' + '=' * 50)
print('真实资金回测关键指标')
print('=' * 50)
for idx, row in metrics_df.iterrows():
    print(f"{row['指标名称']}: {row['指标值']}")

fig.show()

In [ ]:
# 15. 保存回测结果
import os
from datetime import datetime

result_dir = '信号文件'
os.makedirs(result_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
signal_file = f'{result_dir}/首板低开{params_dict["open_pct_low"]}-{params_dict["open_pct_high"]} {timestamp}.csv'
merged_df.to_pandas().to_csv(signal_file, index=False, encoding='utf-8-sig')
print(f'信号文件已保存: {signal_file}')

trade_file = f'{result_dir}/首板低开交易记录 {timestamp}.csv'
交易记录.to_csv(trade_file, index=False, encoding='utf-8-sig')
print(f'交易记录已保存: {trade_file}')